# Сбербанк: формы 0409802 (баланс) и 0409803 (финрезультат)

Порядок работы:
1. **Настройки** — пути и период.
2. **Скачивание** — архивы с сайта ЦБ → строки Сбербанка → `form802_all.parquet`, `form803_all.parquet`.
   Нужно запускать только когда появились новые кварталы.
3. **Таблицы** — сводка «строка × квартал» в млрд руб и выгрузка в Excel.

Что важно знать о данных:
* Формы квартальные. За 2022 год и 1‑е полугодие 2023 ЦБ отчётность не публиковал.
* Шаблон форм менялся, номера строк сдвигались (например, финрезультат в 803 был 8‑й строкой, стал 9‑й).
  Поэтому строки старых периодов сопоставляются с последним отчётом **по названию**.
* 803 — нарастающим итогом с начала года. Квартальные суммы считаются отдельно.

In [ ]:
# 1. Настройки
import io, re, shutil, zipfile
from datetime import date
from pathlib import Path

import pandas as pd
import rarfile
import requests
from dbfread import DBF

rarfile.UNRAR_TOOL = r"C:\Users\Admin\unrar\unrar.exe"
WORKDIR = Path(r"C:\Users\Admin\cbr_data")
PARSED  = WORKDIR / "parsed"
REGN    = 1481                                  # Сбербанк
START, END = date(2020, 1, 1), date(2026, 7, 1)
HEADERS = {"User-Agent": "Mozilla/5.0"}

for sub in ("raw", "extracted", "parsed", "line_names"):
    (WORKDIR / sub).mkdir(parents=True, exist_ok=True)

## 2. Скачивание с сайта ЦБ

In [ ]:
def quarters(start, end):
    """Отчётные даты 01.01, 01.04, 01.07, 01.10 в диапазоне."""
    return [date(y, m, 1) for y in range(start.year, end.year + 1) for m in (1, 4, 7, 10)
            if start <= date(y, m, 1) <= end]


def download(form, d):
    """Архив формы за дату d (кэшируется в raw). None — ЦБ этот период не публиковал."""
    for ext in ("rar", "zip"):
        path = WORKDIR / "raw" / f"{form}-{d:%Y%m%d}.{ext}"
        if path.exists():
            return path
    for ext in ("rar", "zip"):
        path = WORKDIR / "raw" / f"{form}-{d:%Y%m%d}.{ext}"
        r = requests.get(f"https://www.cbr.ru/vfs/credit/forms/{path.name}", headers=HEADERS, timeout=60)
        if r.status_code == 404:
            continue
        r.raise_for_status()
        path.write_bytes(r.content)
        return path
    return None


def unpack(archive):
    """Распаковывает архив в extracted/<имя архива> (один раз)."""
    folder = WORKDIR / "extracted" / archive.stem
    if not any(folder.rglob("*")):
        shutil.rmtree(folder, ignore_errors=True)
        opener = rarfile.RarFile if archive.suffix == ".rar" else zipfile.ZipFile
        with opener(archive) as a:
            a.extractall(folder)
    return folder


def text(v):
    """Текстовое поле DBF → строка без нулевых байтов: '15.5', 'I'."""
    if isinstance(v, bytes):
        v = v.decode("cp866", "ignore")
    return str(v).replace("\x00", "").strip().rstrip(".")


def number(s):
    return pd.to_numeric(s.map(lambda v: text(v) if isinstance(v, bytes) else v), errors="coerce")


def load_form(form, d):
    """Строки Сбербанка из DBF формы 802/803 за дату d. None — архива нет."""
    archive = download(form, d)
    if archive is None:
        return None
    dbf = next(f for f in unpack(archive).rglob("*")
               if f.name.lower().startswith(f"pk{form}") and f.suffix.lower() == ".dbf")
    raw = pd.DataFrame(iter(DBF(dbf, encoding="cp866")))
    raw = raw[number(raw["REGN_GKO"]) == REGN]
    section = raw["RAZDEL"].map(text).replace({"1": "I", "2": "II"}) if "RAZDEL" in raw else ""
    return pd.DataFrame({"report_date": pd.Timestamp(d),
                         "section": section,                 # у 802 разделов нет — пусто
                         "line": raw["STR"].map(text),
                         "total": number(raw["VSEGO"])})      # тыс. руб


def get_names(form, d):
    """Названия строк со страницы этого же отчёта Сбербанка на cbr.ru (кэш в line_names).
    На сайте отчёт на 01.10 лежит под dt = сентябрь."""
    cache = WORKDIR / "line_names" / f"{form}_{d:%Y-%m}.csv"
    if cache.exists():
        return pd.read_csv(cache, dtype=str, keep_default_na=False)
    dt = (pd.Timestamp(d) - pd.DateOffset(months=1)).strftime("%Y%m")
    url = f"https://www.cbr.ru/banking_sector/credit/coinfo/f{form}?regnum={REGN}&dt={dt}"
    html = requests.get(url, headers=HEADERS, timeout=60).text
    table = pd.read_html(io.StringIO(html), match="Наименование показателя", converters={0: str, 1: str})[0]
    rows, section = [], ""
    for num, name in zip(table.iloc[:, 0].astype(str), table.iloc[:, 1].astype(str)):
        header = re.match(r"^\s*(I{1,3}|IV)\.\s", num)          # 'I. АКТИВЫ', 'II. Прочий совокупный доход'
        if header:
            section = header.group(1)
        elif re.fullmatch(r"\d+(\.\d+)*(\(\d+\))?", num.strip()):     # номера вида 15.5 и 11(1)
            name = re.sub(r",?\s*в том числе.*$", "", name.strip())
            rows.append((section if form == "803" else "", num.strip(), name))
    names = pd.DataFrame(rows, columns=["section", "line", "line_name"]).drop_duplicates(["section", "line"])
    names.to_csv(cache, index=False, encoding="utf-8-sig")
    return names

In [ ]:
# Запуск скачивания (несколько минут при первом прогоне, дальше всё из кэша)
for form in ("802", "803"):
    parts = []
    for d in quarters(START, END):
        try:
            df = load_form(form, d)
            if df is None:
                print(f"{form} {d}: архива нет — период не раскрывался")
                continue
            df = df.merge(get_names(form, d), on=["section", "line"], how="left")
            parts.append(df)
            print(f"{form} {d}: {len(df)} строк, без названия: {df['line_name'].isna().sum()}")
        except Exception as e:
            print(f"{form} {d}: ошибка — {type(e).__name__}: {e}")
    pd.concat(parts).to_parquet(PARSED / f"form{form}_all.parquet", index=False)

## 3. Таблицы

`to_latest_numbering` переводит номера строк всех периодов в нумерацию последнего отчёта:
1. если в последнем отчёте под тем же номером то же название — номер не меняется;
2. иначе ищем строку с таким же названием (без учёта регистра и знаков препинания) — берём её номер;
3. не нашли — номер остаётся, а если он уже занят другой строкой, добавляется пометка «(стар.)».

Какие строки переехали, печатается при запуске — это можно проверить глазами.

In [ ]:
def norm_name(s):
    """Название для сравнения: без регистра, знаков препинания и «в том числе»."""
    if not isinstance(s, str):
        return None
    s = s.lower().replace("ё", "е").replace("отчетный год", "отчетный период")
    s = re.sub(r",?\s*в том числе.*$", "", s)
    return re.sub(r"[^\w]+", " ", s).strip()


def to_latest_numbering(df):
    df = df.copy()
    if "section" not in df:                       # файлы от старой версии парсера
        df["section"] = ""
    df["section"] = df["section"].fillna("")
    df["report_date"] = pd.to_datetime(df["report_date"])
    df["key"] = df["line_name"].map(norm_name)

    latest = df[df["report_date"] == df["report_date"].max()]
    same   = set(zip(latest["section"], latest["line"], latest["key"]))
    unique = latest.drop_duplicates(["section", "key"], keep=False)        # названия, встречающиеся один раз
    by_name = dict(zip(zip(unique["section"], unique["key"]), unique["line"]))

    def renumber(g):
        new, taken = {}, set()
        for i, s, l, k in zip(g.index, g["section"], g["line"], g["key"]):          # 1) номер и название совпали
            if (s, l, k) in same:
                new[i] = l; taken.add((s, l))
        for i, s, l, k in zip(g.index, g["section"], g["line"], g["key"]):          # 2) нашли по названию
            target = by_name.get((s, k))
            if i not in new and target and (s, target) not in taken:
                new[i] = target; taken.add((s, target))
        for i, s, l in zip(g.index, g["section"], g["line"]):                       # 3) остальное
            if i not in new:
                new[i] = l if (s, l) not in taken else f"{l} (стар.)"
                taken.add((s, new[i]))
        return pd.Series(new)

    df["line_orig"] = df["line"]
    df["line"] = df.groupby("report_date", group_keys=False).apply(renumber)
    moved = df[df["line"] != df["line_orig"]]
    if len(moved):
        print((moved.groupby(["section", "line_orig", "line"])["report_date"]
                    .agg(с="min", по="max").reset_index().to_string(index=False)))
    return df.drop(columns="key")


def pivot(df):
    """Длинная таблица → (числа «(раздел, строка) × дата», названия строк по последнему отчёту)."""
    t = df.pivot_table(index=["section", "line"], columns="report_date", values="total", aggfunc="first")
    names = (df.sort_values("report_date").dropna(subset=["line_name"])
               .groupby(["section", "line"])["line_name"].last())
    return t, names


def finish(t, names):
    """Убирает пустые столбцы и строки, где всегда 0, сортирует, переводит в млрд руб, добавляет названия."""
    t = t.loc[:, t.notna().any()]
    t = t[(t.fillna(0) != 0).any(axis=1)]
    t = t.loc[sorted(t.index, key=lambda k: (k[0], [int(x) for x in re.findall(r"\d+", k[1])], k[1]))]
    t = t / 1e6                                                          # тыс. руб → млрд руб
    t.insert(0, "line_name", names.reindex(t.index).values)
    t.columns = ["line_name"] + [c.strftime("%Y-%m-%d") for c in t.columns[1:]]
    return t

In [ ]:
# Баланс (802), млрд руб на отчётную дату
t, names = pivot(to_latest_numbering(pd.read_parquet(PARSED / "form802_all.parquet")))

# Часть статей ЦБ не раскрывает (7, 12, 19, 20, 24…), но итоги их включают — считаем остаток
assets      = [("", str(i)) for i in range(1, 14)]
liabilities = [("", str(i)) for i in range(15, 21)]
t.loc[("", "12*"), :] = t.loc[("", "14")] - t.reindex(assets).sum()
t.loc[("", "20*"), :] = t.loc[("", "21")] - t.reindex(liabilities).sum()
names.loc[("", "12*")] = "Нераскрытые активы (стр. 14 минус раскрытые стр. 1–13)"
names.loc[("", "20*")] = "Нераскрытые обязательства (стр. 21 минус раскрытые стр. 15–20)"

balance = finish(t, names).droplevel("section")
balance.round(1)

In [ ]:
# Финрезультат (803), млрд руб
ytd, names = pivot(to_latest_numbering(pd.read_parquet(PARSED / "form803_all.parquet")))

# Квартальные суммы: из итога с начала года вычитаем предыдущий квартал того же года.
# Отчёт на 01.04 — это и есть 1-й квартал. Отчёт на 01.01 — весь прошлый год, из него вычитаем 01.10.
q = ytd.copy()
for col in ytd.columns:
    prev = col - pd.DateOffset(months=3)
    if col.month == 4:
        q[col] = ytd[col]
    elif prev in ytd.columns:
        q[col] = ytd[col] - ytd[prev]
    else:
        q[col] = float("nan")                     # предыдущего квартала нет (2019 и 2022–1п2023)

# ЧПД без корректировок на разницу РВП/ОРОКУ (строки 1.5 и 2.4 заполнялись только до 2024 года)
def row(t, line):
    return t.loc[("I", line)] if ("I", line) in t.index else 0

for t in (ytd, q):
    t.loc[("I", "3*"), :] = (row(t, "1") - row(t, "1.5").fillna(0)) - (row(t, "2") - row(t, "2.4").fillna(0))
names.loc[("I", "3*")] = "ЧПД без корректировок (стр. 1 − 1.5 − (стр. 2 − 2.4))"

pnl_ytd = finish(ytd, names)    # как в отчёте: с начала года
pnl_q   = finish(q, names)      # за квартал
pnl_q.round(1)

In [ ]:
# Выгрузка всех таблиц в Excel
with pd.ExcelWriter(PARSED / "sber_802_803.xlsx") as xw:
    balance.reset_index().to_excel(xw, sheet_name="802 баланс", index=False)
    pnl_q.reset_index().to_excel(xw, sheet_name="803 по кварталам", index=False)
    pnl_ytd.reset_index().to_excel(xw, sheet_name="803 с начала года", index=False)
print("Сохранено:", PARSED / "sber_802_803.xlsx")

## 4. Метрики и rate-volume

* **Доходность кредитов** = проценты по кредитам клиентам (803, стр. 1.2.2) / средние кредиты (802, стр. 4.1.2) × 4.
* **Стоимость средств клиентов** = проценты по ним (803, стр. 2.2) / средние средства юрлиц и физлиц (802, стр. 15.4 + 15.5) × 4.
* **NIM** = ЧПД без корректировок (803, стр. 3*) / средние активы (802, стр. 14) × 4.
* **Rate-volume**: изменение ЧПД между кварталами `D0` и `D1` раскладывается на эффект объёма и эффект ставки по кредитам
  и по средствам клиентов. Остаток («прочее») — ценные бумаги, межбанк, средства ЦБ, облигации.

Средний остаток = (начало квартала + конец) / 2. Кварталы без квартальных процентов (2020‑01, 2023‑07) не считаются.

In [ ]:
# 4. Метрики по кварталам и rate-volume
# Ставки считаются по квартальным процентам (803) и среднему остатку за квартал (802),
# среднее = (остаток на начало квартала + на конец) / 2. Годовые ставки = квартальные × 4.
LOANS = ["4.1.2"]            # кредиты юрлицам и физлицам (по амортизированной стоимости)
FUNDS = ["15.4", "15.5"]     # средства юрлиц и физлиц (по амортизированной стоимости)
ASSETS = ["14"]              # всего активов


def avg_balance(lines, d):
    """Средний остаток строк 802 за квартал, закончившийся датой d (млрд руб). NaN, если нет начала или конца."""
    start = (pd.Timestamp(d) - pd.DateOffset(months=3)).strftime("%Y-%m-%d")
    if start not in balance.columns or d not in balance.columns:
        return float("nan")
    rows = balance.loc[balance.index.intersection(lines)]
    return (rows[start].sum() + rows[d].sum()) / 2


def flow(line, d):
    """Сумма строки раздела I формы 803 за квартал (млрд руб)."""
    return pnl_q.loc[("I", line), d] if ("I", line) in pnl_q.index else float("nan")


m = pd.DataFrame([{
        "quarter_end":     d,
        "loans_avg":       avg_balance(LOANS, d),
        "funds_avg":       avg_balance(FUNDS, d),
        "assets_avg":      avg_balance(ASSETS, d),
        "loan_interest":   flow("1.2.2", d),    # процентные доходы по кредитам клиентам
        "funds_interest":  flow("2.2", d),      # процентные расходы по средствам клиентов
        "nii":             flow("3*", d),       # ЧПД без корректировок
    } for d in pnl_q.columns[1:]]).set_index("quarter_end")

m["loan_yield_%"] = m["loan_interest"] / m["loans_avg"] * 4 * 100
m["funds_cost_%"] = m["funds_interest"] / m["funds_avg"] * 4 * 100
m["spread_%"]     = m["loan_yield_%"] - m["funds_cost_%"]
m["nim_%"]        = m["nii"] / m["assets_avg"] * 4 * 100
m = m.dropna(subset=["loans_avg"])
display(m.round(2))


# ---- Rate-volume: почему ЧПД изменился между двумя кварталами ----
# Проценты за квартал = средний остаток × квартальная ставка, поэтому
#   Δпроцентов = (остаток₁ − остаток₀) × ставка₀   ← эффект объёма
#              + (ставка₁ − ставка₀) × остаток₁    ← эффект ставки
# и сумма эффектов точно равна изменению.
D0, D1 = m.index[-5], m.index[-1]        # по умолчанию: последний квартал против того же квартала год назад


def rate_volume(vol, interest, d0, d1):
    v0, v1 = m.loc[d0, vol], m.loc[d1, vol]
    r0, r1 = m.loc[d0, interest] / v0, m.loc[d1, interest] / v1     # квартальная ставка
    return (v1 - v0) * r0, (r1 - r0) * v1


lv, lr = rate_volume("loans_avg", "loan_interest", D0, D1)
fv, fr = rate_volume("funds_avg", "funds_interest", D0, D1)
d_nii = m.loc[D1, "nii"] - m.loc[D0, "nii"]
rv = pd.Series({
    f"ЧПД {D0}":                         m.loc[D0, "nii"],
    "Кредиты: рост портфеля":            lv,
    "Кредиты: изменение ставок":         lr,
    "Средства клиентов: рост объёма":    -fv,     # больше средств → больше расходов → ЧПД ниже
    "Средства клиентов: изменение ставок": -fr,
    "Прочее (ценные бумаги, межбанк, ЦБ…)": d_nii - (lv + lr - fv - fr),
    f"ЧПД {D1}":                         m.loc[D1, "nii"],
}, name="млрд руб")
display(rv.round(1).to_frame())

m.round(4).to_csv(PARSED / "metrics.csv", encoding="utf-8-sig")
rv.round(2).to_csv(PARSED / "rate_volume.csv", encoding="utf-8-sig")
print("Сохранено: metrics.csv и rate_volume.csv в", PARSED)